# Gothenburg — dynamic traffic simulation

*ECMI Digital City Twin · DTCC Core*

A **dynamic, time-stepped** simulation of the whole city (LWR / Godunov on every road, Lebacque
junctions). Demand comes from the **DeSO zones**: each zone's **centroid** is a source/sink node
that **generates** trips (its registered cars) and **attracts** them (its population), connected to
the road network by a short connector. We march in time and animate four metrics as congestion
builds through the morning peak:

- **load factor** `ρ/ρ_jam` — how full each road is (0 = empty, 1 = jammed),
- **average speed** (km/h),
- **queue length** (m),
- **total network delay**, vehicle-weighted `Σ nₑ·(1 − vₑ/v_free)`, over time (rises into gridlock).

> This replaces the earlier static assignment: it is a genuine simulation with a time loop, so we
> see *when and where* congestion forms — not just an equilibrium snapshot.

In [ ]:
import os, sys, time
sys.path = [p for p in sys.path if p not in ("", ".", os.getcwd())]  # use installed dtcc_core, not local source
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from matplotlib import animation
from IPython.display import HTML

LANE_JAM = 0.15      # jam density per lane (veh/m)
EPS0     = 0.0       # start empty -> "load > 0" means a road is actually used
CFL      = 0.9
EPS      = 1e-9
DX       = 30.0      # UNIFORM cell size (m) for every edge -> uniform Δx and Δt across the network
VMAX_KMH = {"motorway":90,"motorway_link":60,"trunk":70,"primary":60,"secondary":50,
            "secondary_link":50,"tertiary":40,"residential":30,"living_street":20,"unclassified":30}
def vmax_ms(h): return VMAX_KMH.get(str(h), 40)/3.6

## 1. Dynamic model — `Edge` (LWR/Greenshields) + `Node` (Lebacque)

Each road is a chain of cells with a Greenshields fundamental diagram; the Godunov flux moves
vehicles between cells. Junctions use the Lebacque demand–supply rule with turning fractions.
`Edge.metrics()` returns the four quantities we animate.

In [ ]:
class Edge:
    """One directed road (LWR cell chain, Greenshields FD)."""
    def __init__(self, eid, u, v, length, vmax, lanes, dx_target=None):
        self.id=eid; self.u=u; self.v=v; self.vmax=vmax; self.lanes=lanes
        self.rho_jam=LANE_JAM*lanes; self.rho_c=self.rho_jam/2; self.qmax=self.vmax*self.rho_jam/4
        self.N=max(1,int(round(length/DX))); self.dx=DX; self.length=self.N*DX; self.t0=self.length/vmax  # UNIFORM Δx=DX
        self.rho=np.full(self.N, EPS0*self.rho_jam); self.fluxes=np.zeros(self.N+1)
        self.flux_0=0.0; self.flux_N=0.0                      # boundary fluxes (set by junctions)
    def fd(self, r):     return self.vmax*r*(1-r/self.rho_jam)             # Greenshields q(ρ)
    def demand(self, r): return np.where(r<=self.rho_c, self.fd(r), self.qmax)
    def supply(self, r): return np.where(r<=self.rho_c, self.qmax, self.fd(r))
    def demand_at(self,i): return float(self.demand(self.rho[i]))
    def supply_at(self,i): return float(self.supply(self.rho[i]))
    def compute_internal_fluxes(self):
        if self.N<2: return
        self.fluxes[1:self.N]=np.minimum(self.demand(self.rho[:-1]), self.supply(self.rho[1:]))
    def update(self, dt):
        self.fluxes[0]=self.flux_0; self.fluxes[self.N]=self.flux_N    # frozen junction fluxes
        self.rho+=(dt/self.dx)*(self.fluxes[:-1]-self.fluxes[1:])
        np.clip(self.rho, 0.0, self.rho_jam, out=self.rho)
    def metrics(self):
        r=self.rho; avg_rho=r.mean(); avg_q=self.fd(r).mean()
        avg_v=max((self.vmax*(1-r/self.rho_jam)).mean(), EPS)
        tt=self.length/max(avg_v,0.5); delay=max(0.0, tt-self.t0)
        vdelay=avg_rho*self.length*max(0.0, 1-avg_v/self.vmax)   # vehicles x time-fraction lost
        return dict(load=avg_rho/self.rho_jam,           # load factor (0..1)
                    speed=avg_v*3.6,                     # km/h
                    queue=float(np.sum(r>self.rho_c)*self.dx),   # m above critical density
                    delay=delay, flow=avg_q*3600,        # s, veh/h
                    vdelay=vdelay)                       # vehicle-weighted delay (equiv. stopped veh)

class Node:
    """Intersection — Lebacque demand/supply rationing with turning fractions."""
    def __init__(self, nid):
        self.id=nid; self.incoming=[]; self.outgoing=[]; self.tm=[]; self.kind="interior"
    def solve(self):
        if not self.incoming or not self.outgoing or len(self.tm)==0: return
        dem=[e.demand_at(-1) for e in self.incoming]; sup=[e.supply_at(0) for e in self.outgoing]
        od=np.zeros(len(self.outgoing))
        for i in range(len(self.incoming)):
            for j in range(len(self.outgoing)): od[j]+=dem[i]*self.tm[i][j]
        r=np.ones(len(self.outgoing))
        for j in range(len(self.outgoing)):
            if od[j]>sup[j]+EPS: r[j]=sup[j]/od[j]
        for j in range(len(self.outgoing)): self.outgoing[j].flux_0=0.0
        for i in range(len(self.incoming)):
            mr=1.0
            for j in range(len(self.outgoing)):
                if self.tm[i][j]>0: mr=min(mr, r[j])
            q=dem[i]*mr; self.incoming[i].flux_N=q
            for j in range(len(self.outgoing)): self.outgoing[j].flux_0+=q*self.tm[i][j]

## 2. Load Gothenburg and simplify the graph

OSM splits roads at every shape point (edges down to ~0.7 m), which would force a microscopic CFL
step. We merge degree-2 shape points into **super-edges**, then merge super-edges shorter than 35 m,
so the true CFL step is ~1 s and mass is conserved. Free-flow speed uses the **tagged OSM
`maxspeed`** per segment where it exists (~18 % of segments), falling back to a per-class default
otherwise; a super-edge takes the slowest segment along its chain. Every edge is then discretised
with the **same cell size Δx** (each edge = an integer number of Δx-cells), so the time step Δt is
**uniform** across the whole network.

In [ ]:
sys.path = [p for p in sys.path if p not in ("", ".", os.getcwd())]
import dtcc_core as dtcc
NONDRIVE={"footway","steps","pedestrian","path","cycleway","bridleway","track","construction","platform","service","corridor"}
def _num(v,fb):
    if v is None: return fb
    if isinstance(v,(list,tuple,np.ndarray)): v=v[0] if len(v) else None
    try: return float(str(v).strip().split()[0].split(";")[0])
    except: return fb

X0,Y0=319995.962899,6399009.716755; B=dtcc.Bounds(X0-1000,Y0-1000,X0+1000,Y0+1000)
roads=dtcc.datasets.roads(bounds=B)
deso =dtcc.datasets.deso(bounds=B, statistics=["population","cars","employment"])
ra=roads.to_arrays(include_attributes=True)
V0=np.asarray(ra["vertices"],float); E0=np.asarray(ra["edges"]).reshape(-1,2); L0=np.asarray(ra["lengths"],float)
at=ra.get("attributes",{}); hw0=np.asarray(at.get("highway")); ln0=np.asarray(at.get("lanes")); ms0=np.asarray(at.get("maxspeed_kmh"))
keep=np.array([str(h) not in NONDRIVE for h in hw0],bool); E0,L0,hw0,ln0,ms0=E0[keep],L0[keep],hw0[keep],ln0[keep],ms0[keep]
used=np.unique(E0.reshape(-1)); remap=-np.ones(len(V0),int); remap[used]=np.arange(len(used)); Vr=V0[used]; Er=remap[E0]
Hm=nx.MultiGraph(); _n_tagged=0
for i in range(len(Er)):
    msv=_num(ms0[i], np.nan)                                   # tagged maxspeed (km/h) from OSM, or NaN
    if np.isfinite(msv) and msv>0: vseg=msv/3.6; _n_tagged+=1  # use the REAL limit where present
    else:                          vseg=vmax_ms(hw0[i])        # else fall back to the class default
    Hm.add_edge(int(Er[i,0]),int(Er[i,1]),length=float(L0[i]),lanes=max(_num(ln0[i],1),1.),vms=vseg)
print(f"maxspeed: {_n_tagged}/{len(Er)} segments ({100*_n_tagged//max(len(Er),1)}%) use a tagged OSM limit; rest use class defaults")
realset={n for n in Hm.nodes if Hm.degree(n)!=2}; SE=[]; GEOM=[]; seen=set()
for r in realset:
    for nb in list(Hm.neighbors(r)):
        path=[r]; prev=r; cur=nb; Ls=0.; lmin=99; vmin=999
        while True:
            ed=Hm[prev][cur][list(Hm[prev][cur])[0]]
            Ls+=ed["length"]; lmin=min(lmin,ed["lanes"]); vmin=min(vmin,ed["vms"]); path.append(cur)
            if cur in realset: break
            nxts=[x for x in Hm.neighbors(cur) if x!=prev]
            if not nxts: break
            prev,cur=cur,nxts[0]
            if cur in path: break
        a,b=path[0],path[-1]; key=(min(a,b),max(a,b),round(Ls,1))
        if key in seen or a==b: continue
        seen.add(key); SE.append((a,b,Ls,lmin,vmin)); GEOM.append(np.array([Vr[n][:2] for n in path]))
L_MIN=35.0; parent={}
def find(x):
    parent.setdefault(x,x); root=x
    while parent[root]!=root: root=parent[root]
    while parent[x]!=root: parent[x],x=root,parent[x]
    return root
def union(a,b):
    ra,rb=find(a),find(b)
    if ra!=rb: parent[ra]=rb
for (a,b,Ls,lm,vm) in SE:
    if Ls<L_MIN: union(a,b)
SE2={}; GEOM2={}
for (a,b,Ls,lm,vm),g in zip(SE,GEOM):
    ra,rb=find(a),find(b)
    if ra==rb: continue
    key=(min(ra,rb),max(ra,rb))
    if key not in SE2 or Ls>SE2[key][2]: SE2[key]=(ra,rb,Ls,lm,vm); GEOM2[key]=g
SE=list(SE2.values()); GEOM=[GEOM2[k] for k in SE2]
print(f"{len(Er)} raw edges -> {len(SE)} super-edges (min {min(s[2] for s in SE):.0f} m, median {np.median([s[2] for s in SE]):.0f} m)")

## 3. DeSO zone centroids as source/sink nodes + gravity demand

Each DeSO zone centroid becomes a node, joined to its nearest road junction by a short, high-capacity
**connector** (out = production, in = attraction). DeSO zones are whole administrative polygons, so a
zone that only clips the study box has its centroid *outside* it, so we clamp it to the box perimeter
to find a sensible attachment point, then place the source/sink node **on that nearest road junction**
(navy squares sit on the network, where the zone's traffic actually enters/leaves). Trips between zones follow a **gravity model**
`T_ij = O_i · A_j·exp(−γ·t_ij) / Σ_k A_k·exp(−γ·t_ik)`, where production
`O_i = β · population_i` (β = trips generated per resident) and **attraction `A_j` = employment
(jobs)** in zone *j* — a morning commute *home → work*. The term `exp(−γ·t_ij)` is the
travel-probability factor (closer destinations are likelier). Routes are laid onto junction
**turning fractions**, computed once on free-flow travel times.

In [ ]:
# road edges, both directions
edges=[]; eid=0
for (a,b,Ls,lm,vm) in SE:
    edges.append(Edge(eid,a,b,Ls,vm,lm)); eid+=1
    edges.append(Edge(eid,b,a,Ls,vm,lm)); eid+=1
n_road=len(edges)

# DeSO centroids + connectors
dz=deso.to_arrays(); cen=np.asarray(dz["centroids"]); fld=dz["fields"]
z_cars=np.asarray(fld.get("cars_in_traffic"),float).ravel(); z_pop=np.asarray(fld.get("population_total"),float).ravel()
z_emp=np.asarray(fld.get("employed_residents_total"),float).ravel()   # employment -> trip attraction (jobs)
roadnodes=np.array(sorted(realset)); rxy=Vr[roadnodes]; rn_root=np.array([find(int(n)) for n in roadnodes])
present=set(e.u for e in edges)|set(e.v for e in edges)
GEOMc=list(GEOM); centroids=[]; CXY={}
for z in range(len(cen)):
    cz=np.array(cen[z],float); cid=f"Z{z}"
    cz[0]=min(max(cz[0],X0-1000),X0+1000); cz[1]=min(max(cz[1],Y0-1000),Y0+1000)  # clamp to road-box perimeter
    order=np.argsort(np.hypot(rxy[:,0]-cz[0],rxy[:,1]-cz[1]))
    rn=next((int(rn_root[o]) for o in order if int(rn_root[o]) in present), None)
    if rn is None: continue
    d=max(float(np.hypot(Vr[rn][0]-cz[0],Vr[rn][1]-cz[1])),20.0)
    edges.append(Edge(eid,cid,rn,d,13.9,5)); eid+=1                 # OUT connector (production)
    edges.append(Edge(eid,rn,cid,d,13.9,5)); eid+=1                 # IN  connector (attraction)
    GEOMc.append(np.array([cz[:2],Vr[rn][:2]])); GEOMc.append(np.array([Vr[rn][:2],cz[:2]]))
    centroids.append((cid,z,rn)); CXY[cid]=Vr[rn][:2]   # place the node on its access road junction

# network structure
nodes={}
for e in edges:
    nodes.setdefault(e.u,Node(e.u)).outgoing.append(e); nodes.setdefault(e.v,Node(e.v)).incoming.append(e)
for cid,z,rn in centroids: nodes[cid].kind="centroid"
G=nx.DiGraph()
for e in edges: G.add_edge(e.u,e.v,t=e.t0,edge=e)

# gravity OD between centroids -> production rate + turning fractions (static, free-flow routes)
for nd in nodes.values():
    if nd.kind!="centroid" and nd.incoming and nd.outgoing:
        nd.tm=np.zeros((len(nd.incoming),len(nd.outgoing)))
GAMMA, BETA = 0.0008, 0.0003     # GAMMA: distance deterrence; BETA: trips generated per resident
cids=[c[0] for c in centroids]; zof={c[0]:c[1] for c in centroids}
inflow={}
for ci in cids:
    Oi=z_pop[zof[ci]]*BETA                                       # production: BETA * population of zone i
    dist,paths=nx.single_source_dijkstra(G, ci, weight="t")
    dests=[cj for cj in cids if cj!=ci and cj in dist]
    wv=np.array([z_emp[zof[cj]]*np.exp(-GAMMA*dist[cj]) for cj in dests]); pull=wv.sum()  # attraction: employment
    if pull<=0 or Oi<=0: continue
    inflow[ci]=Oi
    for kk,cj in enumerate(dests):
        Tij=Oi*wv[kk]/pull; p=paths[cj]                          # P(i->j) ∝ employment_j · exp(-γ·t_ij)
        for s in range(1,len(p)-1):
            nd=nodes[p[s]]
            if nd.kind=="centroid" or len(nd.tm)==0: continue
            try:
                ii=[e.u for e in nd.incoming].index(p[s-1]); jj=[e.v for e in nd.outgoing].index(p[s+1])
                nd.tm[ii][jj]+=Tij
            except ValueError: pass
for nd in nodes.values():
    if isinstance(nd.tm,np.ndarray) and nd.tm.size:
        for i in range(nd.tm.shape[0]):
            t=nd.tm[i].sum(); nd.tm[i]=nd.tm[i]/t if t>0 else np.ones(nd.tm.shape[1])/nd.tm.shape[1]
        nd.tm=nd.tm.tolist()
print(f"{len(centroids)} DeSO centroids | {len(inflow)} producing | total production {sum(inflow.values())*3600:.0f} cars/h | O=β·population, A=employment | {len(edges)} edges")

## 4. Run the dynamic simulation (time-step loop)

Demand ramps up over the first three minutes (morning peak), then holds. At each snapshot we store
the four metrics, taking the worst of the two directions per super-edge for the maps.

In [ ]:
dt=CFL*min(e.dx/e.vmax for e in edges); T=1200.0; steps=int(T/dt); snap=max(1,steps//40)
sink_conn=[e for e in edges if isinstance(e.v,str)]      # road->centroid connectors absorb trips
nSE=len(SE)
def peak(t): return min(1.0, t/180.0)                    # ramp over 3 min, then hold
times=[]; LOAD=[]; SPEED=[]; QUEUE=[]; TDELAY=[]
t0=time.time()
for k in range(steps):
    pk=peak(k*dt)
    for ci,rate in inflow.items():
        for e in nodes[ci].outgoing: e.flux_0=min(rate*pk, e.supply_at(0))
    for e in sink_conn: e.flux_N=e.demand_at(-1)
    for e in edges: e.compute_internal_fluxes()
    for nd in nodes.values():
        if nd.kind!="centroid": nd.solve()
    for e in edges: e.update(dt)
    if k%snap==0:
        m=[edges[i].metrics() for i in range(n_road)]    # road edges only
        times.append(k*dt)
        LOAD.append(np.array([max(m[2*j]["load"],  m[2*j+1]["load"])  for j in range(nSE)]))
        SPEED.append(np.array([min(m[2*j]["speed"], m[2*j+1]["speed"]) for j in range(nSE)]))
        QUEUE.append(np.array([max(m[2*j]["queue"], m[2*j+1]["queue"]) for j in range(nSE)]))
        TDELAY.append(sum(x["vdelay"] for x in m))   # vehicle-weighted -> rises into gridlock, no false drop
print(f"uniform Δx={DX:.0f} m, uniform Δt={dt:.2f} s, {steps} steps, ran in {time.time()-t0:.0f}s")
print(f"final: max load {LOAD[-1].max():.2f} | jammed (load>0.8): {(LOAD[-1]>0.8).sum()} super-edges | "
      f"min speed {SPEED[-1].min():.1f} km/h | delay {TDELAY[-1]:.0f} veh (equiv. stopped)")

## 5. Animations and plots (each saved as a separate file)

Helper that animates one metric on the real road geometry (navy squares = DeSO centroids) and saves
its own GIF.

In [ ]:
from IPython.display import Image
cxs=[CXY[c][0] for c in CXY]; cys=[CXY[c][1] for c in CXY]
QMAX=max(60.0, float(np.max([q.max() for q in QUEUE])))   # queue colour scale

def animate_map(values, cmap_name, vmax, title, label, fname, thicken=True):
    fig,ax=plt.subplots(figsize=(11,10)); cmap=plt.get_cmap(cmap_name); norm=plt.Normalize(0,vmax)
    lc=LineCollection(GEOM, array=values[0], cmap=cmap, norm=norm, linewidths=1.6, zorder=2)
    ax.add_collection(lc); ax.scatter(cxs,cys,s=26,marker="s",c="navy",zorder=4)
    ax.autoscale(); ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
    plt.colorbar(lc, ax=ax, shrink=0.7, label=label)
    def fr(k):
        lc.set_array(values[k])
        if thicken: lc.set_linewidths(1.0+2.5*np.clip(values[k]/vmax,0,1))
        ax.set_title(f"{title} — t = {times[k]:.0f} s"); return (lc,)
    an=animation.FuncAnimation(fig, fr, frames=len(times), interval=160, blit=False)
    an.save(fname, writer="pillow", fps=7); plt.close(fig)
    print("saved", fname)

### 5a. Load factor `ρ/ρ_jam` — animation (`gothenburg_load.gif`)

In [ ]:
animate_map(LOAD, "RdYlGn_r", 1.0, "Load factor ρ/ρ_jam (0 = empty, 1 = jam)",
            "ρ/ρ_jam", "gothenburg_load.gif")
Image("gothenburg_load.gif")

### 5b. Average speed — animation (`gothenburg_speed.gif`)

In [ ]:
animate_map(SPEED, "RdYlGn", 60.0, "Average speed (km/h, green = fast)",
            "km/h", "gothenburg_speed.gif", thicken=False)
Image("gothenburg_speed.gif")

### 5c. Queue length — animation (`gothenburg_queue.gif`)

In [ ]:
animate_map(QUEUE, "RdYlGn_r", QMAX, "Queue length (m)",
            "m", "gothenburg_queue.gif")
Image("gothenburg_queue.gif")

### 5d. Total network delay — plot (`gothenburg_total_delay.png`)

In [ ]:
fig,ax=plt.subplots(figsize=(8,5))
ax.plot(times, TDELAY, "b", lw=2)
ax.set_xlabel("time (s)"); ax.set_ylabel("delay")
ax.set_title("Total network delay over time"); ax.grid(alpha=0.3)
fig.tight_layout(); fig.savefig("gothenburg_total_delay.png", dpi=110); plt.show()
print("saved gothenburg_total_delay.png")

### 5e. Load-factor histogram at 3 equidistant time moments (`gothenburg_load_histogram.png`)

At three equally-spaced moments (T/3, 2T/3, T) we histogram the load factor over the road
super-edges that are actually used (`load > 0`); each panel's title gives the **number of edges with
load > 0** at that moment.

In [ ]:
moments=[T/3, 2*T/3, T]
idxs=[min(range(len(times)), key=lambda k: abs(times[k]-tt)) for tt in moments]
fig,axs=plt.subplots(1,3,figsize=(15,4.2), sharey=True)
for ax,idx in zip(axs,idxs):
    vals=LOAD[idx]; loaded=vals[vals>0]
    ax.hist(loaded, bins=20, range=(0,1), color="steelblue", edgecolor="white")
    ax.set_title(f"t = {times[idx]:.0f} s\nedges with load > 0:  {loaded.size}")
    ax.set_xlabel("load factor ρ/ρ_jam"); ax.grid(alpha=0.3)
axs[0].set_ylabel("number of edges")
fig.suptitle("Load-factor distribution of loaded edges at 3 equidistant moments", fontsize=13)
fig.tight_layout(); fig.savefig("gothenburg_load_histogram.png", dpi=110); plt.show()
print("saved gothenburg_load_histogram.png")
print("edges with load>0:", {f"{times[i]:.0f}s": int((LOAD[i]>0).sum()) for i in idxs})

## Summary

- **Dynamic city-scale simulation**: LWR/Godunov on every road + Lebacque junctions, marched in
  time (CFL step ≈ 1 s) over the whole simplified Gothenburg network.
- **Demand from DeSO zones**: each zone centroid is a source/sink node that generates trips (its
  β·population) and attracts them (its **jobs/employment**), connected to the road network and routed
  by a gravity model laid onto junction turning fractions.
- **Outputs, each saved as its own file** (simulation length **T = 1200 s**):
  `gothenburg_load.gif`, `gothenburg_speed.gif`, `gothenburg_queue.gif` (animations),
  `gothenburg_total_delay.png` (plot), and `gothenburg_load_histogram.png` (load-factor
  distribution of loaded edges at three equidistant moments T/3, 2T/3, T).
- So we watch congestion **form and spread** during the morning peak, rather than a static
  equilibrium picture.